# Titanic: EDA

How the engineered features in `01_prep` were found. Each section asks one question of the raw data, shows the code that answers it, and ends with what the answer led to.

The questions transfer to any tabular problem:

1. What does each column actually mean?
2. What does each column look like on its own?
3. How does each column relate to the target?
4. Do some columns repeat each other?
5. Does one column's effect depend on another?
6. Is there structure hidden in IDs and text?
7. What does a simple model get wrong?

Section 8 maps every finding to its feature, and section 9 covers how to test new ones.

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

df.shape, df_test.shape

((891, 12), (418, 11))

The tool used most below: the target rate per group, always shown **with the group size**. A 100% rate means little if it comes from 2 passengers.

In [3]:
def survival_by(*keys):
    return df.groupby(list(keys))['Survived'].agg(rate='mean', n='size').round(3)

## 1. What does each column actually mean?

Read the data dictionary, then look at **raw rows grouped by anything that might link them**. Summary statistics hide what a single value stands for.

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# Passengers who share a ticket: what does Fare mean for them?
df[df['Ticket'] == 'CA. 2343'][['Name', 'Pclass', 'Ticket', 'Fare', 'SibSp', 'Parch']]

,Name,Pclass,Ticket,Fare,SibSp,Parch
159,"Sage, Master. Thomas Henry",3,CA. 2343,69.55,8,2
180,"Sage, Miss. Constance Gladys",3,CA. 2343,69.55,8,2
201,"Sage, Mr. Frederick",3,CA. 2343,69.55,8,2
324,"Sage, Mr. George John Jr",3,CA. 2343,69.55,8,2
792,"Sage, Miss. Stella Anna",3,CA. 2343,69.55,8,2
846,"Sage, Mr. Douglas Bullen",3,CA. 2343,69.55,8,2
863,"Sage, Miss. Dorothy Edith ""Dolly""",3,CA. 2343,69.55,8,2


Every Sage family member shows the same fare, 69.55, in 3rd class. **Fare is the price of the whole ticket**, not of one seat. Dividing by the number of passengers on the ticket, counted over train and test together, gives a fare per person. Within a class, it varies far less than the raw fare:

In [6]:
both = pd.concat([df, df_test])
ticket_count = df['Ticket'].map(both['Ticket'].value_counts())

pd.DataFrame({
    'fare_std': df.groupby('Pclass')['Fare'].std(),
    'fare_per_person_std': (df['Fare'] / ticket_count).groupby(df['Pclass']).std(),
}).round(2)

,fare_std,fare_per_person_std
Pclass,,
1,78.38,15.73
2,13.42,2.93
3,11.78,1.61


→ `TicketCount` and `LogFarePP` (log of the fare per person).

## 2. What does each column look like on its own?

Profile every column for three things: **missing values**, **number of distinct values**, and **skew**.

In [7]:
pd.DataFrame({
    'dtype': df.dtypes,
    'missing_train': df.isna().mean().round(3),
    'missing_test': df_test.isna().mean().round(3),
    'distinct': df.nunique(),
})

,dtype,missing_train,missing_test,distinct
Age,float64,0.199,0.206,88
Cabin,str,0.771,0.782,147
Embarked,str,0.002,0.000,3
Fare,float64,0.000,0.002,248
Name,str,0.000,0.000,891
Parch,int64,0.000,0.000,7
PassengerId,int64,0.000,0.000,891
Pclass,int64,0.000,0.000,3
Sex,str,0.000,0.000,2
SibSp,int64,0.000,0.000,7


In [8]:
df.describe().round(2)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.00,891.00,891.00,714.00,891.00,891.00,891.00
mean,446.00,0.38,2.31,29.70,0.52,0.38,32.20
std,257.35,0.49,0.84,14.53,1.10,0.81,49.69
min,1.00,0.00,1.00,0.42,0.00,0.00,0.00
25%,223.50,0.00,2.00,20.12,0.00,0.00,7.91
50%,446.00,0.00,3.00,28.00,0.00,0.00,14.45
75%,668.50,1.00,3.00,38.00,1.00,0.00,31.00
max,891.00,1.00,3.00,80.00,8.00,6.00,512.33


What these say:

- **`Age` (20%) and `Cabin` (77%) are often missing.** Check in section 3 whether *being missing* tells you something.
- **`Name`, `Ticket` and `Cabin` have hundreds of distinct values.** They can't be used as categories directly, but may hide structure (section 6).
- **`Fare` is heavily skewed:** median 14, maximum 512. A few outliers would dominate a linear model, so it gets a log transform.

## 3. How does each column relate to the target?

For categories, look at the rate per value. For numbers, bin them first with `pd.qcut`. Things to look for:

- a **gap between missing and present** rows → add a missing-value flag
- a rate that **rises and then falls** → bin the column, one linear weight can't fit it
- **tiny groups** → merge them, they add noise

In [9]:
survival_by('Sex')

,rate,n
Sex,,
female,0.742,314
male,0.189,577


In [10]:
survival_by('Pclass')

,rate,n
Pclass,,
1,0.630,216
2,0.473,184
3,0.242,491


In [11]:
# Is being missing informative?
pd.concat({
    'Age missing': df.groupby(df['Age'].isna())['Survived'].agg(rate='mean', n='size'),
    'Cabin missing': df.groupby(df['Cabin'].isna())['Survived'].agg(rate='mean', n='size'),
}).round(3)

rate    n
              Age              
Age missing   False  0.406  714
              True   0.294  177
Cabin missing False  0.667  204
              True   0.300  687

Missing rows survived far less often, for both. Missing values aren't random: records were likely less complete for 3rd-class passengers and for those who died. → `AgeMissing` and `HasCabin` flags.

In [12]:
# Does survival rise steadily with family size?
survival_by(df['SibSp'] + df['Parch'] + 1).T

,1,2,3,4,5,6,7,8,11
rate,0.304,0.553,0.578,0.724,0.2,0.136,0.333,0.0,0.0
n,537.000,161.000,102.000,29.000,15.0,22.000,12.000,6.0,7.0


No: 30% alone, 55–72% for families of 2–4, then a sharp drop for larger families. → `FamilyGroup` bins: alone / 2–4 / 5+.

In [13]:
# Numbers: bin into quantiles first.
pd.concat({
    'Age': df.groupby(pd.qcut(df['Age'], 5))['Survived'].mean(),
    'Fare': df.groupby(pd.qcut(df['Fare'], 5))['Survived'].mean(),
}).round(3)

Age   (0.419, 19.0]        0.482
      (19.0, 25.0]         0.328
      (25.0, 31.8]         0.394
      (31.8, 41.0]         0.438
      (41.0, 80.0]         0.373
Fare  (-0.001, 7.854]      0.218
      (7.854, 10.5]        0.201
      (10.5, 21.679]       0.424
      (21.679, 39.688]     0.444
      (39.688, 512.329]    0.642
Name: Survived, dtype: float64

In [14]:
# Deck is the first letter of the cabin: are the groups big enough to learn from?
survival_by(df['Cabin'].str[0].fillna('none')).T

Cabin,A,B,C,D,E,F,G,T,none
rate,0.467,0.745,0.593,0.758,0.75,0.615,0.5,0.0,0.3
n,15.000,47.000,59.000,33.000,32.00,13.000,4.0,1.0,687.0


Apart from "none" (687 passengers), each deck has 4–59 passengers, and deck T has 1. Too thin to learn a rate per deck, and a fold without deck T gets one column fewer. Having a cabin at all is the signal. → `HasCabin` instead of `Deck`.

## 4. Do some columns repeat each other?

Look for columns that are formulas of other columns, highly correlated, or stand-ins for another column. They add no new information, and duplicated inputs make a linear model's weights unstable.

In [15]:
family_size = df['SibSp'] + df['Parch'] + 1

pd.concat([df[['SibSp', 'Parch', 'Pclass', 'Fare', 'Age']], family_size.rename('FamilySize')], axis=1).corr().round(2)

,SibSp,Parch,Pclass,Fare,Age,FamilySize
SibSp,1.00,0.41,0.08,0.16,-0.31,0.89
Parch,0.41,1.00,0.02,0.22,-0.19,0.78
Pclass,0.08,0.02,1.00,-0.55,-0.37,0.07
Fare,0.16,0.22,-0.55,1.00,0.10,0.22
Age,-0.31,-0.19,-0.37,0.10,1.00,-0.30
FamilySize,0.89,0.78,0.07,0.22,-0.30,1.00


`FamilySize` is exactly `SibSp + Parch + 1`, so the three columns (plus `IsAlone`) carry one idea. → Keep only `FamilyGroup`.

In [16]:
# Is Embarked its own signal, or a stand-in for class?
pd.crosstab(df['Embarked'], df['Pclass'], normalize='index').round(2)

Pclass,1,2,3
Embarked,,,
C,0.51,0.10,0.39
Q,0.03,0.04,0.94
S,0.20,0.25,0.55


In [17]:
# Survival by port within each class: the class effect is much stronger than the port effect.
df.pivot_table(index='Embarked', columns='Pclass', values='Survived', aggfunc='mean').round(2)

Pclass,1,2,3
Embarked,,,
C,0.69,0.53,0.38
Q,0.50,0.67,0.38
S,0.58,0.46,0.19


Much of the gap between ports comes from class: half of the passengers from Cherbourg (C) were 1st class, and 94% of those from Queenstown (Q) were 3rd class. Some effect remains within a class (in 3rd class, 19% survival from Southampton vs 38% from C or Q), so dropping it was a judgment call. Removing it didn't lower the CV score. → `Embarked` dropped, and a candidate to add back and test (section 9).

## 5. Does one column's effect depend on another?

Make a cross-table of two strong columns. If the gap from one column **changes size** across the other column, the columns interact.

In [18]:
df.pivot_table(index='Sex', columns='Pclass', values='Survived', aggfunc='mean').round(3)

Pclass,1,2,3
Sex,,,
female,0.968,0.921,0.500
male,0.369,0.157,0.135


Being female adds about 60 points of survival in 1st class, 76 in 2nd, but only 36 in 3rd. A linear model gives `Sex` one weight that applies in every class, so it can't fit this. → `Sex_Pclass`, one category per pair. An MLP can sometimes learn this by itself.

## 6. Is there structure hidden in IDs and text?

Columns with many distinct values often carry meaning in their format. Read samples and look for repeating parts.

In [19]:
df['Name'].sample(8, random_state=1).tolist()

['Swift, Mrs. Frederick Joel (Margaret Welles Barron)',
 'Nenkoff, Mr. Christo',
 'Ilett, Miss. Bertha',
 'Peters, Miss. Katie',
 'Hart, Miss. Eva Miriam',
 'Hansen, Mr. Henry Damsgaard',
 'Navratil, Mr. Michel ("Louis M Hoffman")',
 'Futrelle, Mrs. Jacques Heath (Lily May Peel)']

In [20]:
title = df['Name'].str.extract(r',\s*([^.]*)\.')[0].str.strip()

title.value_counts()

0
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [21]:
# Title knows the age group even when Age is missing.
df.assign(Title=title).groupby('Title')['Age'].agg(['median', 'count', lambda a: a.isna().sum()]).rename(columns={'<lambda_0>': 'missing'}).loc[['Mr', 'Mrs', 'Miss', 'Master']]

,median,count,missing
Title,,,
Mr,30.0,398,119
Mrs,35.0,108,17
Miss,21.0,146,36
Master,3.5,36,4


"Master" marks boys (median age 3.5). Filling every missing age with the overall mean (29.7) turns these boys into adult men. Titles fix two things:

- → `Title`, with rare titles grouped into `Rare` and spelling variants merged (Mlle → Miss).
- → missing ages filled with the **median age for the title**.

The name also contains the surname, which links families. That's used in section 7.

## 7. What does a simple model get wrong?

Fit the simplest reasonable model, then **study only its mistakes**. A feature that explains the mistakes adds information. A feature that explains cases the model already gets right adds nothing.

The simplest rule here: women and boys survive, men die.

In [22]:
is_woman_or_boy = (df['Sex'] == 'female') | (title == 'Master')
pred = is_woman_or_boy.astype(int)

print(f'accuracy: {(pred == df["Survived"]).mean():.3f}')

errors = df[pred != df['Survived']]

errors.groupby(['Sex', 'Pclass']).size().unstack(fill_value=0)

accuracy: 0.793


Pclass,1,2,3
Sex,,,
female,3,6,72
male,42,8,53


The male errors are mostly men who survived (86), plus boys who died (17). Little in the data separates the surviving men from the rest, so they stay mostly unpredictable. The women who died are 89% 3rd class. Count them by surname:

In [23]:
errors.loc[errors['Sex'] == 'female', 'Name'].str.split(',').str[0].value_counts().head(10)

Name
Andersson        5
Palsson          3
Ford             3
Skoog            3
Sage             3
Lefebre          3
Vander Planke    2
Goodwin          2
Zabour           2
Jussila          2
Name: count, dtype: int64

The same families appear again and again: Andersson, Palsson, Ford, Skoog, Sage. **Families lived or died together.** Check that with numbers: in families with several women and boys, how often did all of them share one outcome?

In [24]:
surname = df['Name'].str.split(',').str[0]
family = (surname + '_' + df['Fare'].round(1).astype(str)).where(is_woman_or_boy)

outcomes = df.groupby(family)['Survived'].agg(['mean', 'size'])
multi = outcomes[outcomes['size'] > 1]

print(f'families with 2+ women/boys: {len(multi)}')
print(f'all survived: {(multi["mean"] == 1).mean():.0%}, all died: {(multi["mean"] == 0).mean():.0%}, mixed: {multi["mean"].between(0, 1, inclusive="neither").mean():.0%}')

families with 2+ women/boys: 49
all survived: 63%, all died: 33%, mixed: 4%


Almost every family shared one outcome. → `GroupDied` / `GroupLived`: for each woman or boy, did the *other* labelled women and boys in their family all die, or all survive?

This feature uses labels, so it has to be learned in `Preprocessor.fit` from the training split only, and a passenger's own label must be left out.

## 8. From findings to features

| Finding | Section | Feature | Where it's computed |
|---|---|---|---|
| Fare is per ticket and heavily skewed | 1, 2 | `TicketCount`, `LogFarePP` | `add_features` (no labels used) |
| Missing Age / Cabin predicts lower survival | 3 | `AgeMissing`, `HasCabin` | `add_features` |
| Survival peaks for families of 2–4 | 3 | `FamilyGroup` | `add_features` |
| Decks are too small to learn from | 3 | `HasCabin` replaces `Deck`, `CabinCount` | `add_features` |
| `SibSp`, `Parch`, `FamilySize`, `IsAlone` repeat each other | 4 | dropped, kept as `FamilyGroup` | — |
| `Embarked` stands in for class | 4 | dropped | — |
| Sex effect depends on class | 5 | `Sex_Pclass` | `add_features` |
| Title encodes age group | 6 | `Title`, age filled with the title's median | `add_features`, `Preprocessor.fit` |
| Families lived or died together | 7 | `GroupDied`, `GroupLived` | `Preprocessor.fit` (uses labels) |

Rule of thumb for **where**: a feature computed from a single row (or from columns that use no labels) goes in `add_features`. Anything that learns from data, like a median, a mean or a label rate, goes in `Preprocessor.fit`, so it's learned only from the training split.

## 9. Testing a new feature

- **Change one thing at a time** and re-run `04_k_fold_cv`. Compare the change in score with the spread across folds (about ±0.02–0.03 here). Smaller changes are noise.
- **Trust validation loss more than accuracy.** With ~180 validation rows, accuracy moves in steps of a few passengers.
- **Check for leakage.** Ask: could this be computed for a test passenger without knowing whether they survived? If the feature uses labels, it must be learned per training split and never include the row's own label.
- **Write each pattern down as a sentence** ("boys with no recorded age are filled in as 30-year-olds"). Turning the sentence into code is usually the easy part.